# PoolFormer on Kaggle — two-GPU DDP

Select a **two-GPU accelerator** in Kaggle's notebook settings, attach the CIFAR-100 Python dataset, then run these cells in order. All model and training code is embedded below; no extra files or pip installs are needed.

Defaults: PoolFormer-S12, CIFAR-100, 224px, 20 epochs for a practical first run. The paper uses 300 ImageNet epochs; change `EPOCHS` if desired. Training includes AdamW, warmup/cosine decay, MixUp/CutMix, AutoAugment, random erasing, label smoothing, stochastic depth and LayerScale.

Two processes are launched with [PyTorch torchrun](https://docs.pytorch.org/docs/stable/elastic/run). Batch size is **per GPU**: 32 × 2 = 64 globally. tqdm displays batch loss and evaluation accuracy on rank zero; epoch metrics combine both GPUs. CIFAR-100 uses 45,000 training / 5,000 validation images, then evaluates the best model once on the 10,000-image official test set.


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("Visible GPUs:", torch.cuda.device_count())
for index in range(torch.cuda.device_count()):
    print(f"GPU {index}: {torch.cuda.get_device_name(index)}")
if torch.cuda.device_count() < 2:
    raise RuntimeError("Select a two-GPU accelerator in Kaggle notebook settings, then restart the session.")


PyTorch: 2.10.0+cu128
CUDA available: True
Visible GPUs: 2
GPU 0: Tesla T4
GPU 1: Tesla T4


## Configuration

Set `DATA_ROOT` to the parent of `cifar-100-python` or that directory itself. For an ImageNet-style dataset, set `DATASET = "imagenet"` and use a root containing `train/<class>` and `val/<class>` folders.

Set `SMOKE_TEST = True` to check the two-GPU launch on synthetic 32px images for one epoch before a full run. Each launch gets a fresh output directory so rerunning does not overwrite checkpoints. Keep `WARMUP_EPOCHS < EPOCHS`. If GPU memory is insufficient, lower `BATCH_SIZE_PER_GPU`.


In [2]:
from datetime import datetime, timezone
from pathlib import Path

DATASET = "cifar100"
DATA_ROOT = "/kaggle/input/datasets/nhatxminh2603/cifar-100-python"
VARIANT = "s12"                 # s12, s24, s36, m36, m48
EPOCHS = 100                    # paper: 300
WARMUP_EPOCHS = 5
BATCH_SIZE_PER_GPU = 32         # global batch size = 64
IMAGE_SIZE = 224               # 32 is faster for CIFAR, with smaller feature maps
WORKERS_PER_GPU = 2
LEARNING_RATE = None           # default: 0.001 * global_batch_size / 1024
SEED = 42
SMOKE_TEST = False
DOWNLOAD = True               # CIFAR only; requires Kaggle Internet enabled

WORKING = Path("/kaggle/working")
WORKING.mkdir(parents=True, exist_ok=True)
# For download mode, use a writable root such as /kaggle/working/data.
if not SMOKE_TEST and not DOWNLOAD:
    root = Path(DATA_ROOT)
    required = [root / "train", root / "test", root / "meta"] if root.name == "cifar-100-python" else [root / "cifar-100-python" / name for name in ("train", "test", "meta")]
    if DATASET == "imagenet":
        required = [root / "train", root / "val"]
    missing = [str(path) for path in required if not path.exists()]
    if missing:
        raise FileNotFoundError("Attach the dataset and update DATA_ROOT. Missing: " + ", ".join(missing))
print("Configured global batch size:", BATCH_SIZE_PER_GPU * 2)


Configured global batch size: 64


## PoolFormer architecture

The four S12 stages have channels `[64, 128, 320, 512]` and block counts `[2, 2, 6, 2]`. Each block uses GroupNorm → (3×3 average pool − input) → scaled residual, then GroupNorm → 1×1 channel MLP with GELU → scaled residual. The classifier uses final normalization, global average pooling and a linear head.

This cell writes the same implementation as the project's `poolformer.py`.


In [3]:
%%writefile /kaggle/working/poolformer.py
import torch
import torch.nn as nn
from torchvision.ops import StochasticDepth

# PoolFormer.pdf, Tables 1 and 6: depths, widths, peak drop rate, LayerScale.
VARIANTS = {
    "s12": ((2, 2, 6, 2), (64, 128, 320, 512), 0.1, 1e-5),
    "s24": ((4, 4, 12, 4), (64, 128, 320, 512), 0.1, 1e-5),
    "s36": ((6, 6, 18, 6), (64, 128, 320, 512), 0.2, 1e-6),
    "m36": ((6, 6, 18, 6), (96, 192, 384, 768), 0.3, 1e-6),
    "m48": ((8, 8, 24, 8), (96, 192, 384, 768), 0.4, 1e-6),
}

class StarReLU(nn.Module):
    def __init__(self, scale: float = 1.0, bias: float = 0.0):
        super().__init__()
        self.s = nn.Parameter(torch.tensor(scale))
        self.b = nn.Parameter(torch.tensor(bias))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.s * torch.relu(x).square() + self.b

class PoolFormerBlock(nn.Module):
    def __init__(self, dim: int, pool_size: int = 3, mlp_ratio: float = 4.0, act_layer=nn.GELU, ls_init: float = 1e-5, drop_path: float = 0.0):
        super().__init__()
        if pool_size < 1 or pool_size % 2 == 0:
            raise ValueError("pool_size must be a positive odd integer")
        self.norm1 = nn.GroupNorm(1, dim)
        self.pool = nn.AvgPool2d(pool_size, stride=1, padding=pool_size // 2, count_include_pad=False)
        self.norm2 = nn.GroupNorm(1, dim)
        hidden = int(dim * mlp_ratio)
        self.mlp = nn.Sequential(
            nn.Conv2d(dim, hidden, 1),
            act_layer(),
            nn.Conv2d(hidden, dim, 1),
        )
        self.s1 = nn.Parameter(ls_init * torch.ones(dim, 1, 1))
        self.s2 = nn.Parameter(ls_init * torch.ones(dim, 1, 1))
        self.drop_path = StochasticDepth(drop_path, mode="row")

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        normalized = self.norm1(x)
        x = x + self.drop_path(self.s1 * (self.pool(normalized) - normalized))
        x = x + self.drop_path(self.s2 * self.mlp(self.norm2(x)))
        return x

class PoolFormer(nn.Module):
    def __init__(self, layers=(2, 2, 6, 2), dims=(64, 128, 320, 512), num_classes=1000, act_layer=nn.GELU, drop_path_rate=0.1, ls_init=1e-5):
        super().__init__()
        if len(layers) != 4 or len(dims) != 4 or any(n < 1 for n in (*layers, *dims)):
            raise ValueError("layers and dims must each contain four positive integers")
        if num_classes < 1 or not 0 <= drop_path_rate < 1:
            raise ValueError("num_classes must be positive and drop_path_rate must be in [0, 1)")
        self.stages = nn.ModuleList()
        in_c = 3
        patch_specs = [(7, 4, 2), (3, 2, 1), (3, 2, 1), (3, 2, 1)]
        block_index = 0
        for (k, s, p), d, num_b in zip(patch_specs, dims, layers):
            stage = [nn.Conv2d(in_c, d, kernel_size=k, stride=s, padding=p)]
            stage += [PoolFormerBlock(
                d, act_layer=act_layer, ls_init=ls_init,
                drop_path=drop_path_rate * (block_index + i) / max(sum(layers) - 1, 1),
            ) for i in range(num_b)]
            self.stages.append(nn.Sequential(*stage))
            in_c = d
            block_index += num_b
        self.norm = nn.GroupNorm(1, dims[-1])
        self.head = nn.Linear(dims[-1], num_classes)
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for stage in self.stages:
            x = stage(x)
        return self.head(self.norm(x).mean(dim=[-2, -1]))

# Standard variants: S12, S24, S36, M36, M48
def create_poolformer(variant="s12", num_classes=1000):
    layers, dims, drop_rate, ls_init = VARIANTS[variant]
    return PoolFormer(layers, dims, num_classes, drop_path_rate=drop_rate, ls_init=ls_init)


def poolformer_s12(num_classes=1000): return create_poolformer("s12", num_classes)
def poolformer_s24(num_classes=1000): return create_poolformer("s24", num_classes)
def poolformer_s36(num_classes=1000): return create_poolformer("s36", num_classes)
def poolformer_m36(num_classes=1000): return create_poolformer("m36", num_classes)
def poolformer_m48(num_classes=1000): return create_poolformer("m48", num_classes)

if __name__ == "__main__":
    m = poolformer_s12()
    out = m(torch.randn(2, 3, 224, 224))
    assert out.shape == (2, 1000)
    print(f"PoolFormer-S12 OK: {sum(p.numel() for p in m.parameters()) / 1e6:.1f}M params")


Writing /kaggle/working/poolformer.py


## DDP trainer

The script uses [DistributedDataParallel](https://docs.pytorch.org/docs/stable/generated/torch.nn.parallel.DistributedDataParallel.html) with one process per GPU and a DistributedSampler for training. Validation/test shards are disjoint and unpadded; all-reduced totals give exact dataset metrics. Only rank zero emits progress bars and writes checkpoints. CUDA training uses mixed precision.

The embedded trainer also runs on one device outside this notebook. No `module.` prefixes are stored in checkpoints, so a saved model can be loaded without DDP.


In [4]:
%%writefile /kaggle/working/train_poolformer.py
"""PoolFormer training: CIFAR-100 adaptation or ImageNet folders; see README.md."""
import argparse
import math
import os
import random
from pathlib import Path

import torch
import torch.distributed as dist
from torch import nn
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Subset
from torch.utils.data.distributed import DistributedSampler
from torchvision import datasets, transforms
from torchvision.transforms import v2
from tqdm.auto import tqdm

from poolformer import VARIANTS, create_poolformer


def make_datasets(args):
    cifar = args.dataset == "cifar100"
    mean = (0.5071, 0.4867, 0.4408) if cifar else (0.485, 0.456, 0.406)
    std = (0.2675, 0.2565, 0.2761) if cifar else (0.229, 0.224, 0.225)
    policy = transforms.AutoAugmentPolicy.CIFAR10 if cifar else transforms.AutoAugmentPolicy.IMAGENET
    train_transform = transforms.Compose([
        transforms.RandomResizedCrop(args.image_size, interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.RandomHorizontalFlip(), transforms.AutoAugment(policy),
        transforms.ToTensor(), transforms.Normalize(mean, std),
        transforms.RandomErasing(p=0.25),
    ])
    val_transform = transforms.Compose([
        transforms.Resize(round(args.image_size / 0.875), interpolation=transforms.InterpolationMode.BICUBIC),
        transforms.CenterCrop(args.image_size), transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    if args.smoke_test:
        return (datasets.FakeData(4, (3, args.image_size, args.image_size), 100, train_transform),
                datasets.FakeData(5, (3, args.image_size, args.image_size), 100, val_transform, random_offset=4),
                None)
    if cifar:
        root = Path(args.data)
        if root.name == "cifar-100-python" and (root / "train").exists():
            args.data = str(root.parent)
        train = datasets.CIFAR100(args.data, train=True, download=args.download, transform=train_transform)
        val = datasets.CIFAR100(args.data, train=True, transform=val_transform)
        indices = torch.randperm(len(train), generator=torch.Generator().manual_seed(args.seed)).tolist()
        split = len(train) // 10
        test = datasets.CIFAR100(args.data, train=False, download=args.download, transform=val_transform)
        return Subset(train, indices[split:]), Subset(val, indices[:split]), test
    train = datasets.ImageFolder(Path(args.data) / "train", transform=train_transform)
    val = datasets.ImageFolder(Path(args.data) / "val", transform=val_transform)
    if train.class_to_idx != val.class_to_idx:
        raise ValueError("train/ and val/ must have matching class folders")
    return train, val, None


def run_epoch(model, loader, criterion, device, optimizer=None, mixer=None, scaler=None):
    training = optimizer is not None
    model.train(training)
    # Evaluation shards may have unequal lengths; bypass DDP forward collectives.
    forward_model = model.module if not training and isinstance(model, DDP) else model
    rank = dist.get_rank() if dist.is_initialized() else 0
    loss_sum = correct = count = 0
    with torch.set_grad_enabled(training):
        progress = tqdm(loader, desc="Train" if training else "Evaluate", leave=False, disable=rank != 0)
        for images, labels in progress:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            hard_labels = labels
            if training:
                images, labels = mixer(images, labels)
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
                logits = forward_model(images)
                loss = criterion(logits, labels)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            loss_sum += loss.item() * images.size(0)
            # Accuracy is meaningful on the unaugmented validation data only.
            if not training:
                correct += (logits.argmax(1) == hard_labels).sum().item()
            count += images.size(0)
            if rank == 0:
                metrics = {"loss": f"{loss_sum / count:.4f}"}
                if not training:
                    metrics["accuracy"] = f"{100 * correct / count:.2f}%"
                progress.set_postfix(metrics)
    totals = torch.tensor([loss_sum, correct, count], dtype=torch.float64, device=device)
    if dist.is_initialized():
        dist.all_reduce(totals)
    loss_sum, correct, count = totals.tolist()
    return loss_sum / count, 100 * correct / count


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--dataset", choices=("cifar100", "imagenet"), default="cifar100")
    parser.add_argument("--data", default="data")
    parser.add_argument("--variant", choices=VARIANTS, default="s12")
    parser.add_argument("--epochs", type=int, default=300)
    parser.add_argument("--batch-size", type=int, default=64, help="Batch size per GPU/process")
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--workers", type=int, default=4)
    parser.add_argument("--warmup-epochs", type=int, default=5)
    parser.add_argument("--lr", type=float, help="Default: 0.001 * global_batch_size / 1024")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--output", type=Path, default=Path("runs/poolformer"))
    parser.add_argument("--download", action="store_true", help="Download CIFAR-100 if needed")
    parser.add_argument("--smoke-test", action="store_true", help="One epoch on synthetic data at 32px")
    args = parser.parse_args(argv)
    if args.smoke_test:
        args.epochs, args.batch_size, args.image_size, args.workers, args.warmup_epochs = 1, 2, 32, 0, 0
        torch.set_num_threads(2)
    if min(args.epochs, args.batch_size, args.image_size) < 1 or args.workers < 0:
        parser.error("epochs, batch-size, image-size must be positive; workers must be nonnegative")
    if not 0 <= args.warmup_epochs < args.epochs:
        parser.error("warmup-epochs must be nonnegative and less than epochs")
    if args.lr is not None and (not math.isfinite(args.lr) or args.lr <= 0):
        parser.error("lr must be finite and positive")
    # A new run must not silently replace another run's checkpoints.
    if any((args.output / name).exists() for name in ("best.pt", "last.pt")):
        parser.error("output already contains checkpoints; choose a new --output directory")
    distributed = int(os.environ.get("WORLD_SIZE", "1")) > 1
    local_rank = int(os.environ.get("LOCAL_RANK", "0"))
    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        device = torch.device("cuda", local_rank)
    else:
        device = torch.device("cpu")
    if distributed:
        dist.init_process_group("nccl" if device.type == "cuda" else "gloo")
    rank = dist.get_rank() if distributed else 0
    world_size = dist.get_world_size() if distributed else 1
    random.seed(args.seed + rank)
    torch.manual_seed(args.seed + rank)
    if distributed and args.download and args.dataset == "cifar100" and not args.smoke_test:
        if rank == 0:
            datasets.CIFAR100(args.data, train=True, download=True)
        dist.barrier()
        args.download = False
    train, val, test = make_datasets(args)
    num_classes = 100 if args.dataset == "cifar100" or args.smoke_test else len(train.classes)
    loader_args = dict(batch_size=args.batch_size, num_workers=args.workers,
                       pin_memory=device.type == "cuda", persistent_workers=args.workers > 0)
    train_sampler = DistributedSampler(train, seed=args.seed) if distributed else None
    train_loader = DataLoader(train, shuffle=train_sampler is None, sampler=train_sampler, **loader_args)
    # Disjoint evaluation shards, with no padding/duplicate samples.
    if distributed:
        val = Subset(val, range(rank, len(val), world_size))
    val_loader = DataLoader(val, **loader_args)
    model = create_poolformer(args.variant, num_classes).to(device)
    if distributed:
        model = DDP(model, device_ids=[local_rank] if device.type == "cuda" else None)
    base_model = model.module if distributed else model
    # Bias, normalization affine parameters and LayerScale receive no decay.
    decay = [p for name, p in base_model.named_parameters() if p.ndim > 1 and not name.endswith(("s1", "s2"))]
    no_decay = [p for name, p in base_model.named_parameters() if p.ndim <= 1 or name.endswith(("s1", "s2"))]
    lr = args.lr if args.lr is not None else 1e-3 * args.batch_size * world_size / 1024
    optimizer = torch.optim.AdamW([
        {"params": decay, "weight_decay": 0.05},
        {"params": no_decay, "weight_decay": 0.0},
    ], lr=lr)

    def lr_factor(epoch):
        if epoch < args.warmup_epochs:
            return (epoch + 1) / args.warmup_epochs
        progress = (epoch - args.warmup_epochs) / (args.epochs - args.warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * min(progress, 1)))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    mixer = v2.RandomChoice([v2.MixUp(alpha=0.8, num_classes=num_classes),
                             v2.CutMix(alpha=1.0, num_classes=num_classes)])
    train_loss = nn.CrossEntropyLoss(label_smoothing=0.1)
    val_loss = nn.CrossEntropyLoss()
    if rank == 0:
        args.output.mkdir(parents=True, exist_ok=True)
    best_acc = -1.0
    if rank == 0:
        print(f"PoolFormer-{args.variant.upper()} | {device} | world size={world_size} | "
              f"global batch={args.batch_size * world_size} | "
              f"{sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters | peak lr={lr:g}")
    for epoch in range(args.epochs):
        if train_sampler is not None:
            train_sampler.set_epoch(epoch)
        current_lr = optimizer.param_groups[0]["lr"]
        loss, _ = run_epoch(model, train_loader, train_loss, device, optimizer, mixer, scaler)
        eval_loss, accuracy = run_epoch(model, val_loader, val_loss, device)
        scheduler.step()
        improved = accuracy > best_acc
        best_acc = max(best_acc, accuracy)
        if rank == 0:
            checkpoint = dict(model=base_model.state_dict(), optimizer=optimizer.state_dict(),
                              scheduler=scheduler.state_dict(), scaler=scaler.state_dict(),
                              epoch=epoch + 1, best_accuracy=best_acc, val_loss=eval_loss, variant=args.variant,
                              num_classes=num_classes, world_size=world_size,
                              config={**vars(args), "output": str(args.output)})
            for name in (["last.pt", "best.pt"] if improved else ["last.pt"]):
                temporary = args.output / (name + ".tmp")
                torch.save(checkpoint, temporary)
                temporary.replace(args.output / name)
            print(f"Epoch {epoch + 1}/{args.epochs} | lr={current_lr:g} | train loss={loss:.4f} | "
                  f"val loss={eval_loss:.4f} | val accuracy={accuracy:.2f}%", flush=True)
        if distributed:
            dist.barrier()
    if test is not None:
        checkpoint = torch.load(args.output / "best.pt", map_location=device, weights_only=True)
        base_model.load_state_dict(checkpoint["model"])
        if distributed:
            test = Subset(test, range(rank, len(test), world_size))
        _, accuracy = run_epoch(model, DataLoader(test, **loader_args), val_loss, device)
        if rank == 0:
            print(f"Best checkpoint CIFAR-100 test accuracy: {accuracy:.2f}%")
    if rank == 0:
        print(f"Best validation accuracy: {best_acc:.2f}% | checkpoints: {args.output}")
    if distributed:
        dist.destroy_process_group()


if __name__ == "__main__":
    try:
        main()
    finally:
        if dist.is_initialized():
            dist.destroy_process_group()


Writing /kaggle/working/train_poolformer.py


## Launch training

The live output includes tqdm train/validation batch bars, per-epoch global metrics and checkpoint locations. tqdm's running batch metrics are local to GPU 0; the printed epoch metrics cover both GPUs. Interrupting this cell terminates both training workers.


In [5]:
import codecs
import os
import shlex
import subprocess
import sys

run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
OUTPUT = WORKING / ("poolformer_" + VARIANT + ("_smoke_" if SMOKE_TEST else "_") + run_id)
command = [
    sys.executable, "-u", "-m", "torch.distributed.run",
    "--standalone", "--nproc_per_node=2",
    str(WORKING / "train_poolformer.py"),
    "--dataset", DATASET, "--data", DATA_ROOT, "--variant", VARIANT,
    "--epochs", str(EPOCHS), "--warmup-epochs", str(WARMUP_EPOCHS),
    "--batch-size", str(BATCH_SIZE_PER_GPU), "--image-size", str(IMAGE_SIZE),
    "--workers", str(WORKERS_PER_GPU), "--seed", str(SEED), "--output", str(OUTPUT),
]
if LEARNING_RATE is not None:
    command += ["--lr", str(LEARNING_RATE)]
if DOWNLOAD:
    command += ["--download"]
if SMOKE_TEST:
    command += ["--smoke-test"]
print(shlex.join(command), flush=True)

# Forward bytes to Jupyter while preserving tqdm carriage returns.
# A separate process group lets Interrupt stop torchrun and both workers.
import signal
process = subprocess.Popen(command, cwd=WORKING, stdout=subprocess.PIPE,
                           stderr=subprocess.STDOUT, start_new_session=True, bufsize=0)
decoder = codecs.getincrementaldecoder("utf-8")(errors="replace")
try:
    while True:
        chunk = os.read(process.stdout.fileno(), 4096)
        if not chunk:
            break
        sys.stdout.write(decoder.decode(chunk))
        sys.stdout.flush()
    sys.stdout.write(decoder.decode(b"", final=True))
    return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
except BaseException:
    if process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
    raise
finally:
    process.stdout.close()


/usr/bin/python3 -u -m torch.distributed.run --standalone --nproc_per_node=2 /kaggle/working/train_poolformer.py --dataset cifar100 --data /kaggle/input/datasets/nhatxminh2603/cifar-100-python --variant s12 --epochs 100 --warmup-epochs 5 --batch-size 32 --image-size 224 --workers 2 --seed 42 --output /kaggle/working/poolformer_s12_20261005_085133_566881 --download

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
[W1005 08:51:35.010806590 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1005 08:51:44.770375565 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W1005 08:51:44.770708309 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
/usr/

## Inspect and load the best checkpoint

Download `best.pt` / `last.pt` from the output directory in Kaggle's output files. Both include model and optimizer state, scheduler, scaler, completed epoch, configuration and best validation accuracy. Full-run CIFAR-100 test accuracy is printed above. Smoke-test accuracy is from synthetic data.


In [6]:
import importlib

sys.path.insert(0, str(WORKING))
import poolformer
importlib.reload(poolformer)

checkpoint_path = OUTPUT / "best.pt"
checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
model = poolformer.create_poolformer(checkpoint["variant"], checkpoint["num_classes"])
model.load_state_dict(checkpoint["model"])
model.eval()
print("Best checkpoint:", checkpoint_path)
print("Completed epoch:", checkpoint["epoch"])
print("DDP world size:", checkpoint["world_size"])
print(f"Best validation accuracy: {checkpoint['best_accuracy']:.2f}%")
print("Files:", [path.name for path in OUTPUT.iterdir()])


Best checkpoint: /kaggle/working/poolformer_s12_20261005_085133_566881/best.pt
Completed epoch: 91
DDP world size: 2
Best validation accuracy: 59.50%
Files: ['best.pt', 'last.pt']
